## SWE - nanobind

One way to get more performance is to write the kernel in C++ and bind it back to Python. This notebook takes the binding-library and FFI (foreign function interface) approach with **nanobind**, the modern successor to pybind11.

### Table of Contents

1. [Imports and source layout](#sec1)
2. [The C++ source](#sec2)
3. [Build with nanobind + CMake](#sec3)
4. [Acceptance and timing](#sec4)
5. [nanobind vs pybind11](#sec5)
6. [Limitation: binding and build glue](#sec6)

### <a id="sec1"></a>1. Imports and source layout

The C++ source lives at `notebooks/swe_step.cpp`. We invoke `cmake` and
the system C++ compiler via `subprocess`. The resulting shared library is
imported back into Python.

---

**Quick Docs**

- `nanobind.cmake_dir()`: path to nanobind's CMake config files. We pass
  it via `-Dnanobind_DIR=` so `find_package(nanobind CONFIG REQUIRED)`
  succeeds.
- `nanobind_add_module(target source)`: nanobind's CMake helper that
  creates a Python extension target with the right include paths,
  link flags, and visibility settings.
- `nb::ndarray<T, nb::ndim<1>>` (on the C++ side): a typed NumPy view (`double*` + `shape`).
  No copy unless the layout demands one.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import nanobind
import psutil

import swe_core

# One OpenMP thread per physical core, pinned, as in NB 10.
os.environ.setdefault('OMP_NUM_THREADS', str(psutil.cpu_count(logical=False)))
os.environ.setdefault('OMP_PROC_BIND', 'close')
os.environ.setdefault('OMP_PLACES', 'cores')

# Shared problem parameters.
N       = 16384
L       = 10.0
H0      = 1.0
AMP     = 0.1
SIG     = 0.5
CFL     = 0.4
G       = 9.81
N_STEPS = 1000
dx      = L / N
DT      = swe_core.fixed_dt(H0 + AMP, dx, cfl=CFL, g=G)

# CMakeLists.txt + swe_step.cpp live next to swe_core.py; build artifacts go in build/.
CPP_DIR   = Path(swe_core.SWE_STEP_CPP).parent
BUILD_DIR = CPP_DIR / 'build'

# Float64 NumPy reference for validation
h_ref, _ = swe_core.solve_numpy(N, N_STEPS)

### <a id="sec2"></a>2. The C++ source

Same **Read / Compute / Update** shape as NB 08, expressed in C++ via nanobind:

- **Read**. `double*` pointer nanobind hands us, a zero-copy view of the NumPy buffer.
- **Compute**. One flux per interface, written into a caller-provided face buffer, no temporaries.
- **Update**. Writes into `double*`. No Python objects on the hot path.

From `swe_step.cpp`:

```cpp
#include <nanobind/nanobind.h>
#include <nanobind/ndarray.h>
#include <cmath>
#include <algorithm>

namespace nb = nanobind;

inline void rusanov_face(double hL, double hR, double huL, double huR,
                         double g, double& Fh, double& Fhu) {
    /* ...same arithmetic as swe_core.step_numpy... */
}

// Each interface flux is computed exactly once, as in swe_core.step_numpy.
void cpp_step(
    nb::ndarray<const double, nb::ndim<1>, nb::c_contig> h_in,
    nb::ndarray<const double, nb::ndim<1>, nb::c_contig> hu_in,
    nb::ndarray<double, nb::ndim<1>, nb::c_contig> h_out,
    nb::ndarray<double, nb::ndim<1>, nb::c_contig> hu_out,
    nb::ndarray<double, nb::ndim<1>, nb::c_contig> Fh_buf,
    nb::ndarray<double, nb::ndim<1>, nb::c_contig> Fhu_buf,
    double dx, double dt, double g)
{
    const double* h = h_in.data();
    /* ... */
    // Pass 1: one flux per interface i+1/2, i = 0..N.
    #pragma omp parallel for
    for (size_t f = 0; f <= N; ++f)
        rusanov_face(h[f], h[f + 1], hu[f], hu[f + 1], g, Fh[f], Fhu[f]);

    // Pass 2: difference the stored fluxes over the interior.
    #pragma omp parallel for
    for (size_t i = 1; i <= N; ++i) { /* ... */ }
}

NB_MODULE(swe_step, m) {
    m.def("cpp_step", &cpp_step,
          nb::arg("h"), nb::arg("hu"), nb::arg("h_new"), nb::arg("hu_new"),
          nb::arg("Fh"), nb::arg("Fhu"),
          nb::arg("dx"), nb::arg("dt"), nb::arg("g") = 9.81);
}
```

Things to note:
- `nb::ndarray<const double, nb::ndim<1>, nb::c_contig>` is a zero-copy typed view into the NumPy buffer; `nb::c_contig` rejects strided views
- The face-flux buffers are pre-allocated by NumPy and passed in, so the kernel allocates nothing
- `NB_MODULE`'s name must match the compiled library filename (`swe_step.cpython-...so`)
- `nb::arg` carries Python keyword names and defaults into the generated signature.
- Both passes run under `#pragma omp parallel for`, the same directives used with PyOMP. Each thread writes its own index range, no synchronisation needed.

### <a id="sec3"></a>3. Build with nanobind + CMake

nanobind ships with a CMake helper, `nanobind_add_module(...)`, that
sets up include paths, link flags, RPATH, and visibility for you. Let's
write a small `CMakeLists.txt` and then configure and build the library.

---

**Quick Docs**

- `cmake -B build -S .` configures (out-of-source build directory).
- `-DCMAKE_BUILD_TYPE=Release` is essential for the compiler to emit optimisation flags (`-O3 -DNDEBUG`).
- `-DPython_EXECUTABLE=` ensures we correctly link against the Python executable in our venv.
- `cmake --build build --config Release -j` compiles our code into a shared library.

In [ ]:
# locate nanobind's CMake config
NANOBIND_CMAKE_DIR = nanobind.cmake_dir()

CMAKELISTS = '''
cmake_minimum_required(VERSION 3.20)
project(swe_step LANGUAGES CXX)
set(CMAKE_CXX_STANDARD 17)
find_package(Python 3.12 REQUIRED COMPONENTS Interpreter Development.Module)
find_package(nanobind CONFIG REQUIRED)
find_package(OpenMP REQUIRED)
nanobind_add_module(swe_step swe_step.cpp)
target_compile_options(swe_step PRIVATE -O3 -march=native)
target_link_libraries(swe_step PRIVATE OpenMP::OpenMP_CXX)
'''
(CPP_DIR / 'CMakeLists.txt').write_text(CMAKELISTS)

# TODO: configure + build the C++ extension with swe_core.run_cmd(..., cwd=CPP_DIR).
#   - cmake -B build -S . with these flags:
#       -DCMAKE_BUILD_TYPE=Release           (so g++ gets -O3)
#       -DPython_EXECUTABLE=<sys.executable> (the venv interpreter)
#       -Dnanobind_DIR=NANOBIND_CMAKE_DIR    (so find_package(nanobind) works)
#   - cmake --build build --config Release -j
...

Now that we have our shared library, let's import it on the Python side:

In [ ]:
sys.path.insert(0, str(BUILD_DIR))
import swe_step as cpp_swe

### <a id="sec4"></a>4. Acceptance and timing

The time loop reuses pre-allocated state and face-flux buffers, swapping
input and output each step (double buffering), so the timing measures the
kernel call, not per-step allocation. Boundary conditions are re-applied on
the Python side (`swe_core.apply_bc_reflective`) between steps, so the C++
kernel only does the work we're benchmarking.

In [ ]:
def run_nanobind() -> tuple[np.ndarray, np.ndarray]:
    h, hu = swe_core.bump_ic(N, L=L, h0=H0, amplitude=AMP, sigma=SIG)
    h2 = np.empty_like(h); hu2 = np.empty_like(hu)
    Fh, Fhu = np.empty(N + 1), np.empty(N + 1)   # one slot per interface
    for _ in range(N_STEPS):
        swe_core.apply_bc_reflective(h, hu)
        cpp_swe.cpp_step(h, hu, h2, hu2, Fh, Fhu, dx, DT, G)
        h, hu, h2, hu2 = h2, hu2, h, hu
    return h, hu

# No cold/warm split here: compile cost was paid at build time and the dlopen
# at import, so every function call runs warm.
h_nb, _ = run_nanobind()

warm = swe_core.timed_run(run_nanobind, warmup=2, repeats=5, label='11_nanobind')

diff = swe_core.max_diff(h_ref, h_nb)
swe_core.report_and_verify(warm, diff, tol=1e-12, n=N, steps=N_STEPS)

npy_ms = 1e3 * next(r['median_s'] for r in swe_core.load_timings() if r['tool'] == 'numpy')
print(f"NumPy baseline {npy_ms:.1f} ms -> {npy_ms / (1e3 * warm['median_s']):.2f}x")

swe_core.save_timing(
    warm, grid_str=f'N={N}', tool='nanobind', hardware='cpu',
    dtype='float64', steps=N_STEPS,
    max_diff_vs_numpy=diff, binding='nanobind',
    flags='-O3 -march=native',
)

### <a id="sec5"></a>5. nanobind vs pybind11

pybind11 is nanobind's predecessor and remains the most widely used C++↔Python binding library (PyTorch, SciPy, and much of the HPC ecosystem). The ergonomics are nearly identical: in pybind11 the array type is `py::array_t<double, py::array::c_style>` and the entry point is `PYBIND11_MODULE`. nanobind's published benchmarks report up to ~10× lower per-call overhead, ~4× faster compilation, and ~5× smaller binaries ([nanobind](https://github.com/wjakob/nanobind)).

### <a id="sec6"></a>6. Limitation: binding and build glue

This notebook's cost is everything around the kernel, not the kernel itself:

- **The bindings track the C++ API.** `NB_MODULE`, `m.def`, and `nb::arg` are a separate layer that has to be hand-edited whenever the C++ source evolves. The kernel is also no longer regular C++, it carries nanobind types like `nb::ndarray`.
- **Build system.** Building the module requires working with CMake and a C++ toolchain.
- **Recompile-on-edit cycle.** Editing the source leads to recompilation, restarting the kernel and re-importing the module.
- **FFI boundary cost.** Each call from Python into C++ pays a small fixed overhead for argument conversion. At ~1000 calls per run it stays well under a millisecond, negligible against the kernel time.

With `-O3 -march=native` the compiler still emits scalar code for this
loop, yet Sec. 4 lands well ahead of single-threaded NumPy: no Python
overhead, no temporaries, and both passes are multi-threaded.

**EXTRA CREDIT:** make the kernel even faster by changing only compile flags. The cell
below rebuilds the same source as module `swe_step_fast`. Edit
its `target_compile_options` line and re-run. The first build imports cleanly; a loaded C extension cannot be
reloaded without a kernel restart. Run `swe_core.report_and_verify` to ensure correctness. What is the largest gain you can reach, and which
flag does the work?

---

**Quick Docs**

- `target_compile_options(swe_step_fast PRIVATE <flags>)`: the line to
  extend. Candidates: `-ffast-math`, `-fno-math-errno`, `-funroll-loops`.

In [ ]:
# EXTRA CREDIT: rebuild with extra flags and measure the gain vs -O3 -march=native.
assert 'swe_step_fast' not in sys.modules, 'restart the kernel before rebuilding with new flags'

# Same source, second module name, extra flags.
(CPP_DIR / 'swe_step_fast.cpp').write_text(
    open(swe_core.SWE_STEP_CPP).read().replace('NB_MODULE(swe_step,', 'NB_MODULE(swe_step_fast,'))

CMAKELISTS = '''
cmake_minimum_required(VERSION 3.20)
project(swe_step_fast LANGUAGES CXX)
set(CMAKE_CXX_STANDARD 17)
find_package(Python 3.12 REQUIRED COMPONENTS Interpreter Development.Module)
find_package(nanobind CONFIG REQUIRED)
find_package(OpenMP REQUIRED)
nanobind_add_module(swe_step_fast swe_step_fast.cpp)
# TODO: extend the line below with flags that vectorise; keep the gate green
target_compile_options(swe_step_fast PRIVATE -O3 -march=native)
target_link_libraries(swe_step_fast PRIVATE OpenMP::OpenMP_CXX)
'''
(CPP_DIR / 'CMakeLists.txt').write_text(CMAKELISTS)

swe_core.run_cmd(['cmake', '-B', 'build', '-S', '.',
                  '-DCMAKE_BUILD_TYPE=Release',
                  f'-DPython_EXECUTABLE={sys.executable}',
                  f'-Dnanobind_DIR={NANOBIND_CMAKE_DIR}'], cwd=CPP_DIR)
swe_core.run_cmd(['cmake', '--build', 'build', '--config', 'Release', '-j'], cwd=CPP_DIR)

import swe_step_fast as cpp_fast

def run_fast(n_cells=N, n_steps=N_STEPS):
    dx_ = L / n_cells
    dt_ = swe_core.fixed_dt(H0 + AMP, dx_, cfl=CFL, g=G)
    h, hu = swe_core.bump_ic(n_cells, L=L, h0=H0, amplitude=AMP, sigma=SIG)
    h2 = np.empty_like(h); hu2 = np.empty_like(hu)
    Fh, Fhu = np.empty(n_cells + 1), np.empty(n_cells + 1)
    for _ in range(n_steps):
        swe_core.apply_bc_reflective(h, hu)
        cpp_fast.cpp_step(h, hu, h2, hu2, Fh, Fhu, dx_, dt_, G)
        h, hu, h2, hu2 = h2, hu2, h, hu
    return h, hu

h_fast, _ = run_fast()
warm_fast = swe_core.timed_run(run_fast, warmup=2, repeats=5, label='11_nanobind')
diff_fast = swe_core.max_diff(h_ref, h_fast)
swe_core.report_and_verify(warm_fast, diff_fast, tol=1e-12, n=N, steps=N_STEPS)
print(f"{warm['median_s'] / warm_fast['median_s']:.1f}x vs the -O3 -march=native build")

# The synthesis notebook compares each tool at its best: replace the row.
if diff_fast < 1e-12:
    swe_core.save_timing(
        warm_fast, grid_str=f'N={N}', tool='nanobind', hardware='cpu',
        dtype='float64', steps=N_STEPS, max_diff_vs_numpy=diff_fast,
        binding='nanobind', flags='-O3 -march=native')
    swe_core.save_sweep('11_nanobind', run_fast)

In [ ]:
# Rates across the shared size range, for the synthesis notebook (14).
# Pick up faster implementation if EXTRA CREDIT was done.
_mod = cpp_fast if 'cpp_fast' in globals() else cpp_swe

def run_nanobind_at(n_cells, n_steps):
    dx_ = L / n_cells
    dt_ = swe_core.fixed_dt(H0 + AMP, dx_, cfl=CFL, g=G)
    h, hu = swe_core.bump_ic(n_cells, L=L, h0=H0, amplitude=AMP, sigma=SIG)
    h2, hu2 = np.empty_like(h), np.empty_like(hu)
    Fh, Fhu = np.empty(n_cells + 1), np.empty(n_cells + 1)
    for _ in range(n_steps):
        swe_core.apply_bc_reflective(h, hu)
        _mod.cpp_step(h, hu, h2, hu2, Fh, Fhu, dx_, dt_, G)
        h, hu, h2, hu2 = h2, hu2, h, hu
    return h

swe_core.save_sweep('11_nanobind', run_nanobind_at)

---

**Recap.**

- A short `CMakeLists.txt` + `nb::ndarray<...>` exposes a C++ kernel to Python with zero-copy NumPy arrays.
- The cost is the glue around the kernel: a bindings layer to maintain as the C++ evolves, plus build and FFI overhead.
- The kernel computes each face once, as `step_numpy` does, with both passes under `#pragma omp parallel for`. `-O3 -march=native` is still scalar yet well ahead of NumPy; one more flag vectorises it ~2.4x further (EXTRA CREDIT).

Next: `12__swe__cppjit__cub.ipynb` uses JIT-compiled C++/CUDA, with no CMake, recompilation, or separate shared library. It provides automatic Python bindings and runs the whole solve on the GPU using the CUB library.